In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import numpy as np


# LOAD DATA

df = pd.read_csv("questionnaire_data_other_factors.csv")



def merge_groups(g):
    return pd.Series({
        "association": np.average(g["association"], weights=g["n"]),
        "n": g["n"].sum()
    })

# --- NUMERIC + BINARY (same structure, no value column)
num_bin = df[df["type"].isin(["numeric", "binary"])].copy()

num_bin = (
    num_bin
    .groupby(["score", "factor", "type"], as_index=False)
    .apply(merge_groups)
    .reset_index(drop=True)
)

# --- CATEGORICAL (IMPORTANT: includes value column)
cat = df[df["type"] == "categorical"].copy()

cat = (
    cat
    .groupby(["score", "factor", "value"], as_index=False)
    .apply(merge_groups)
    .reset_index(drop=True)
)

cat["type"] = "categorical"

# RECOMBINE CLEANED DATA

df = pd.concat([num_bin, cat], ignore_index=True)

# OUTPUT DIRECTORY

OUT_DIR = Path("questionnaire_diagrams_other_factors")
OUT_DIR.mkdir(exist_ok=True)

plt.rcParams["figure.figsize"] = (14, 10)

# LABEL FUNCTION 

def add_value_labels(bars, values):

    for bar, text in zip(bars, values):

        width = bar.get_width()

        if width >= 0:
            x = width
            ha = "left"
        else:
            x = width
            ha = "right"

        plt.text(
            x,
            bar.get_y() + bar.get_height() / 2,
            text,
            va="center",
            ha=ha,
            fontsize=8
        )

# PLOTTING 

for questionnaire in sorted(df["score"].unique()):

    print(f"Processing {questionnaire}")

    qdf = df[df["score"] == questionnaire]

    # NUMERIC

    numeric = qdf[qdf["type"] == "numeric"].copy()

    if len(numeric) > 0:

        numeric = numeric.sort_values("association", ascending=True)

        labels = numeric["factor"]

        annotation = [
            f"r={a:.3f}\nn={n}"
            for a, n in zip(numeric["association"], numeric["n"])
        ]

        plt.figure(figsize=(14, max(8, len(numeric) * 0.35)))

        bars = plt.barh(labels, numeric["association"])

        plt.axvline(0, linestyle="--")

        add_value_labels(bars, annotation)

        plt.title(f"{questionnaire} - Numeric Factors")
        plt.xlabel("Spearman Correlation")
        plt.tight_layout()

        plt.savefig(
            OUT_DIR / f"{questionnaire}_numeric.png",
            dpi=300
        )

        plt.close()

    # BINARY

    binary = qdf[qdf["type"] == "binary"].copy()

    if len(binary) > 0:

        binary = binary.sort_values("association", ascending=True)

        labels = binary["factor"]

        annotation = [
            f"r={a:.3f}\nn={n}"
            for a, n in zip(binary["association"], binary["n"])
        ]

        plt.figure(figsize=(14, max(8, len(binary) * 0.35)))

        bars = plt.barh(labels, binary["association"])

        plt.axvline(0, linestyle="--")

        add_value_labels(bars, annotation)

        plt.title(f"{questionnaire} - Binary Factors")
        plt.xlabel("Correlation")
        plt.tight_layout()

        plt.savefig(
            OUT_DIR / f"{questionnaire}_binary.png",
            dpi=300
        )

        plt.close()

    # CATEGORICAL

    categorical = qdf[qdf["type"] == "categorical"].copy()

    if len(categorical) > 0:

        categorical["label"] = (
            categorical["factor"].astype(str)
            + " = "
            + categorical["value"].astype(str)
        )

        categorical = categorical.sort_values(["factor", "value"])

        labels = categorical["label"]

        annotation = [
            f"{a:.3f}\nn={n}"
            for a, n in zip(categorical["association"], categorical["n"])
        ]

        plt.figure(figsize=(16, max(10, len(categorical) * 0.28)))

        bars = plt.barh(labels, categorical["association"])

        plt.axvline(0, linestyle="--")

        add_value_labels(bars, annotation)

        plt.title(f"{questionnaire} - Categorical Factors")
        plt.xlabel("Difference from Questionnaire Mean")
        plt.tight_layout()

        plt.savefig(
            OUT_DIR / f"{questionnaire}_categorical.png",
            dpi=300
        )

        plt.close()

print("\nDONE")
print(f"Saved to: {OUT_DIR}")

C:\Users\veron\AppData\Local\Temp\ipykernel_12584\982057208.py:28: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(merge_groups)
C:\Users\veron\AppData\Local\Temp\ipykernel_12584\982057208.py:38: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(merge_groups)


Processing chiq_result
Processing dass_depression
Processing dass_fear
Processing dass_stress
Processing gvas_result
Processing midas_result
Processing pgic_result

DONE
Saved to: questionnaire_diagrams
